# Prescriptions for the Stochasticity Effect on the Integrated X-ray Luminosity of Star-forming Galaxies

## Implications for Selecting Star-forming Galaxies and AGN in X-ray Surveys

This notebook demonstrates how to apply the best-fit prescriptions presented in the associated study in order to estimate the stochasticity-driven variation of the integrated X-ray luminosity of star-forming galaxies.

The implemented models reproduce the lower and upper Highest Density Intervals (HDIs) of the total X-ray luminosity produced by high-mass X-ray binaries (HMXBs) for different confidence intervals:

- 68%
- 90%
- 99%
- 99.9%

The notebook provides:
- utilities for loading galaxy catalogues,
- functions for applying the stochasticity prescriptions,
- examples using a sample catalogue,
- and export functionality for saving the predicted luminosity bounds.

---

## Required Inputs

The input catalogue should contain at least the following columns:

| Column | Description |
|---|---|
| `SFR` | Star formation rate |
| `Metal` | Metallicity |

---

## Repository Contents

The notebook assumes the following files are available in the same directory:

- `Upper_bound_HDIs_best_fit_results.csv`
- `Low_bound_HDIs_best_fit_results.csv`
- `example_sample.csv`



## 1. Imports and Utility Functions

This section imports the required Python packages and defines helper functions used throughout the notebook.

No modifications are required here unless you wish to adapt the notebook to a different catalogue format or workflow.


In [1]:
import pandas as pd
import numpy as np
from astropy.table import Table
import csv
import joblib
import os

In [2]:
# Function that loads the catalog (file) that contains the galaxies to be classified
def load_file(name, file_format):
    """
    Load a file based on the given format.

    Parameters:
        name (str): The base name of the file (without extension).
        file_format (str): The format of the file, either 'csv' or 'fits'.

    Returns:
        pandas.DataFrame
    """
    if file_format=='csv':
        df = pd.read_csv(name+'.csv')
    elif file_format=='fits':
        dat = Table.read(name+'.fits', format='fits')
        df = dat.to_pandas()
    return df



## Function for the fifth-order polynomial of two variables
def poly5_xy(inputs, c00, c01, c02, c03, c04, c05, c10, c11, c12, c13, c14, c20, c21, c22, c23, c30, c31, c32, c40, c41, c50):
    """
    Evaluate the fifth-order two-dimensional polynomial.

    The polynomial is expressed as a function of:
    - metallicity (X)
    - logarithm of the star formation rate log10(Y)

    Parameters
    ----------
    inputs : tuple
        Tuple containing:
        - X : float
            Metallicity value.
        - Y : float
            Star formation rate (SFR).
    
    c00 ... c50 : float
        Polynomial coefficients defining the best-fit surface.

    Returns
    -------
    float
        Evaluated polynomial value corresponding to log10(Lx).

    Notes
    -----
    The implemented polynomial has the form:

    f(X, logY) = Σ c_ij X^i logY^j

    where:
    - i + j <= 5
    - logY = log10(Y)
    """    
    
    
    # Unpack the input variables
    X, Y = inputs
    # Compute the logarithm of the SFR
    logY = np.log10(Y)
    
    return( c00 + c01*logY + c02*logY**2 + c03*logY**3 + c04*logY**4 + c05*logY**5 + 
            c10*X + c11*X*logY + c12*X*logY**2 + c13*X*logY**3 + c14*X*logY**4 + 
            c20*X**2 + c21*X**2*logY + c22*X**2*logY**2 + c23*X**2*logY**3 +
            c30*X**3 + c31*X**3*logY + c32*X**3*logY**2 +
            c40*X**4 + c41*X**4*logY + 
            c50*X**5
          )

## Function for the derivation of the expected total X-ray luminosity for the upper 68%,90%,99%, and 99.9% HDI
def Lx_stochasticity_calclulation_upper_bounds(X,Y,coeffs_df,conf_interval):
    """
    Compute the upper stochasticity luminosity bound.

    This function evaluates the best-fit polynomial prescription
    corresponding to the upper Highest Density Interval (HDI)
    boundary for a given confidence interval.

    Parameters
    ----------
    X : float
        Metallicity value.
    
    Y : float
        Star formation rate (SFR).

    coeffs_df : pandas.DataFrame
        DataFrame containing the polynomial coefficients for
        all confidence intervals.

    conf_interval : int
        Confidence interval percentage.
        Supported values are:
        - 68
        - 90
        - 99
        - 999

    Returns
    -------
    float
        Predicted upper log10(X-ray luminosity) bound.
    """    
    
    # Extract the coefficients for the selected confidence interval     
    coeffs = coeffs_df[f'{conf_interval}_upb'].values
    
    # Evaluate the polynomial prescription
    logLx = poly5_xy((X,Y), *coeffs)
           
    return(logLx)

## Function for the derivation of the expected total X-ray luminosity for the lower 68%,90%,99%, and 99.9% HDI
def Lx_stochasticity_calclulation_lower_bounds(X,Y,coeffs_df,conf_interval):
    """
    Compute the lower stochasticity luminosity bound.

    This function evaluates the best-fit polynomial prescription
    corresponding to the lower Highest Density Interval (HDI)
    boundary for a given confidence interval.

    Parameters
    ----------
    X : float
        Metallicity value.
    
    Y : float
        Star formation rate (SFR).

    coeffs_df : pandas.DataFrame
        DataFrame containing the polynomial coefficients for
        all confidence intervals.

    conf_interval : int
        Confidence interval percentage.
        Supported values are:
        - 68
        - 90
        - 99
        - 999

    Returns
    -------
    float
        Predicted lower log10(X-ray luminosity) bound.
    """    
    
    # Extract the coefficients for the selected confidence interval
    coeffs = coeffs_df[f'{conf_interval}_lob'].values
    
    # Evaluate the polynomial prescription
    logLx = poly5_xy((X,Y), *coeffs)
    return(logLx)



# Function to save the file with the galaxies containing their activity classification as a separate column
def save_cat(df_c, cat_name, cat_format):
    """
    Save a catalog to a file in the specified format.

    Parameters:
        cat_name (str): The name of the file to be saved (without extension).
        cat_format (str): The format in which the catalog should be saved. Supported formats: 'fits' and 'csv'.

    Returns:
        None
    """
    if (cat_format=='fits'):
        new_t = Table.from_pandas(df_c)
        new_t.write(cat_name+'.fits')
    elif (cat_format=='csv'): 
        df_c.to_csv(cat_name+'.csv',index=False)
    else :
        return


## 2. Load an Input Catalogue

As an example, we load the provided `example_sample.csv` catalogue.

You may replace this file with your own catalogue in either:
- CSV format
- FITS format

The catalogue should contain the required columns described above.


In [4]:
# Load the example catalogue
df_data = load_file(name='example_sample', file_format='csv')

# Display the catalogue
display(df_data)

,PGC,SFR,Metal
0,9279,0.263627,8.502527
1,10966,0.127186,8.370684
2,12607,0.342623,8.532990
3,13317,0.869736,8.622491
4,14448,1.399048,8.747322
...,...,...,...
72,3441671,0.237185,8.144732
73,3662384,0.367829,8.415813
74,4078746,0.003786,8.054772
75,4080350,0.165747,8.365657


## 3. Load the Best-fit Coefficients

The following tables contain the best-fit coefficients used to estimate the stochasticity-driven lower and upper luminosity bounds.


In [5]:
df_upper_surface = load_file('Upper_bound_HDIs_best_fit_results',file_format='csv')
df_lower_surface = load_file('Low_bound_HDIs_best_fit_results',file_format='csv')

## 4. Apply the Prescriptions

In this section, the stochasticity prescriptions are applied to each galaxy in the catalogue.

For every source, the notebook computes the predicted lower and upper luminosity bounds for multiple confidence intervals.


The application workflow is wrapped into a reusable function so that users can easily apply the prescriptions to arbitrary galaxy catalogues without modifying the notebook internals.


In [8]:
def apply_stochasticity_prescriptions(
    df,
    metallicity_column='Metal',
    sfr_column='SFR',
    bounds_list=[68, 90, 99, 999],
    verbose=True
):
    """
    Apply the stochasticity prescriptions to an input galaxy catalogue.

    Parameters
    ----------
    df : pandas.DataFrame
        Input galaxy catalogue.
    metallicity_column : str, optional
        Column containing metallicity values.
    sfr_column : str, optional
        Column containing star formation rate values.
    bounds_list : list, optional
        List of confidence intervals to evaluate.
    verbose : bool, optional
        If True, print the computed bounds for each galaxy.

    Returns
    -------
    pandas.DataFrame
        Updated catalogue containing the predicted lower and upper
        luminosity bounds for all requested confidence intervals.
    """

    # Create a copy of the catalogue
    df_output = df.copy()

    # Extract arrays
    X_array = df_output[metallicity_column].values
    Y_array = df_output[sfr_column].values

    # Iterate through all galaxies
    for counter, (X, Y) in enumerate(zip(X_array, Y_array)):

        if verbose:
            print(f'Galaxy {counter}')
            print(f'SFR = {np.round(Y, 4)}')
            print(f'Metallicity = {np.round(X, 4)}')
            print('--' * 25)

        for CI in bounds_list:

            # Compute lower bound
            logLx_stoch_low = Lx_stochasticity_calclulation_lower_bounds(
                X,
                Y,
                coeffs_df=df_lower_surface,
                conf_interval=CI
            )

            # Compute upper bound
            logLx_stoch_up = Lx_stochasticity_calclulation_upper_bounds(
                X,
                Y,
                coeffs_df=df_upper_surface,
                conf_interval=CI
            )

            # Store results in the catalogue
            df_output.loc[counter, f'logLx_lower_{CI}'] = logLx_stoch_low
            df_output.loc[counter, f'logLx_upper_{CI}'] = logLx_stoch_up

            if verbose:
                print(f'Confidence Interval: {CI}%')
                print(f'Lower bound = {np.round(logLx_stoch_low, 4)}')
                print(f'Upper bound = {np.round(logLx_stoch_up, 4)}')
                print()

    return df_output


In [9]:
# Apply the stochasticity prescriptions
df_results = apply_stochasticity_prescriptions(
    df_data,
    metallicity_column='Metal',
    sfr_column='SFR',
    bounds_list=[68, 90, 99, 999],
    verbose=True
)

# Display the updated catalogue
display(df_results)

Galaxy 0
SFR = 0.2636
Metallicity = 8.5025
--------------------------------------------------
Confidence Interval: 68%
Lower bound = 37.675
Upper bound = 39.0337

Confidence Interval: 90%
Lower bound = 37.3846
Upper bound = 39.9233

Confidence Interval: 99%
Lower bound = 37.1255
Upper bound = 40.3407

Confidence Interval: 999%
Lower bound = 36.8547
Upper bound = 40.5625

Galaxy 1
SFR = 0.1272
Metallicity = 8.3707
--------------------------------------------------
Confidence Interval: 68%
Lower bound = 36.9448
Upper bound = 38.8065

Confidence Interval: 90%
Lower bound = 36.7485
Upper bound = 39.6728

Confidence Interval: 99%
Lower bound = 36.3427
Upper bound = 40.205

Confidence Interval: 999%
Lower bound = 35.8625
Upper bound = 40.451

Galaxy 2
SFR = 0.3426
Metallicity = 8.533
--------------------------------------------------
Confidence Interval: 68%
Lower bound = 37.9222
Upper bound = 39.1473

Confidence Interval: 90%
Lower bound = 37.6138
Upper bound = 39.9991

Confidence Interval:

,PGC,SFR,Metal,logLx_lower_68,logLx_upper_68,logLx_lower_90,logLx_upper_90,logLx_lower_99,logLx_upper_99,logLx_lower_999,logLx_upper_999
0,9279,0.263627,8.502527,37.674992,39.033725,37.384586,39.923258,37.125486,40.340748,36.854734,40.562506
1,10966,0.127186,8.370684,36.944776,38.806536,36.748539,39.672769,36.342671,40.205021,35.862483,40.451020
2,12607,0.342623,8.532990,37.922231,39.147318,37.613817,39.999075,37.354903,40.396951,37.132580,40.604716
3,13317,0.869736,8.622491,38.666965,39.627374,38.382945,40.203299,38.080326,40.596359,37.924599,40.761810
4,14448,1.399048,8.747322,38.890253,39.791565,38.686502,40.206404,38.414028,40.639410,38.252761,40.806595
...,...,...,...,...,...,...,...,...,...,...,...
72,3441671,0.237185,8.144732,37.824097,39.395381,37.455128,40.183677,37.050407,40.453209,36.767599,40.576560
73,3662384,0.367829,8.415813,38.085513,39.333291,37.737231,40.139832,37.425689,40.473584,37.210354,40.640036
74,4078746,0.003786,8.054772,42.557864,32.682854,37.529945,36.184430,18.128073,38.761505,14.108832,39.930855
75,4080350,0.165747,8.365657,37.245265,38.949249,36.997093,39.832756,36.660033,40.282941,36.282364,40.497200


## 5. Export the Catalogue

After computing the luminosity bounds, the updated catalogue can be exported in either CSV or FITS format.


In [12]:
# Export the updated catalogue

save_cat(df_c=df_results,
    cat_name='results',
    cat_format='fits')

save_cat(df_c=df_results,
         cat_name='/your_path/catalog_name',
    'csv'
)


## Citation

If you use this notebook or the associated prescriptions in your work, please cite the corresponding publication.

---

## Notes

- The notebook is designed for reproducibility and easy adaptation to user catalogues.
- The implementation assumes that all required coefficient tables are available locally.
- Users are encouraged to adapt the workflow for large survey catalogues or automated pipelines.
